This is our multi step algorithm to determine best locations for a coffee shop given our knowledge graph

1. Build a Node Regression Pipeline: 
    - Target Output: Avg_rating of a BusinessLocation Node
    - Input Features: Block Group attributes, Business Location Node locations, BusLoc - BusLoc relationships)

2. Extract Weights from Node Regression

3. Generate an "Optimal" Block Group

4. Similarity Search to Find Real Block Groups Closest to the Optimal Block Group
    - Rank by closeness in vector space 

5. Generate 10 New Sample Locations to Test
    - Use the geographical distribution of the businesses and county boundaries 
    - Test against zone locations to auto disqualify

6. Input the Generated Sample Locations into the Knowledge Graph

7. Use Node Regression Pipeline to Return Avg_Rating
    - Rank locations based on rating and other criteria


### 1. Setup

In [3]:
import pandas as pd

# import geopandas as gpd

import os

from dotenv import load_dotenv
from decimal import Decimal
from neo4j import GraphDatabase


In [4]:



group_driver = GraphDatabase.driver(
     "bolt://67.58.49.87:7687",
     auth=("neo4j", "h2u9l4px")
)


with group_driver.session() as session:
    result = session.run("MATCH (n) UNWIND labels(n) AS label RETURN count(DISTINCT label) AS count")
    num_nodes = result.single()["count"]
    print(f"Connection Successful: {num_nodes} unique node types found in the graph database")


Connection Successful: 11 unique node types found in the graph database


### 2. Best Location Algorithm

#### 2.1 Node Regression

##### Delete Existing Graph Projections and Pipelines

In [53]:
delete_pipeline_query = """
CALL gds.pipeline.list() 
YIELD pipelineName
CALL gds.pipeline.drop(pipelineName) 
YIELD pipelineName AS droppedPipeline
RETURN 'Dropped pipeline: ' + droppedPipeline AS Result


"""

with group_driver.session() as session:
    result = session.run(delete_pipeline_query)
    for record in result:
        print(record)




In [54]:


delete_graph_projection_query = """
CALL gds.graph.list() 
YIELD graphName
CALL gds.graph.drop(graphName) 
YIELD graphName AS droppedGraph
RETURN 'Dropped projected graph: ' + droppedGraph AS Result

"""

with group_driver.session() as session:
    result = session.run(delete_graph_projection_query)
    for record in result:
        print(record)




In [55]:





delete_models_query = """
CALL gds.model.list() 
YIELD modelName
CALL gds.model.drop(modelName) 
YIELD modelName AS droppedModel
RETURN 'Dropped GDS model: ' + droppedModel AS Result

"""

with group_driver.session() as session:
    result = session.run(delete_models_query)
    for record in result:
        print(record)




##### 2.1.1 Configure Pipeline

In [56]:
pipeline_query = """ 
CALL gds.alpha.pipeline.nodeRegression.create('pipe')

YIELD name, nodePropertySteps, featureProperties, splitConfig, autoTuningConfig, parameterSpace
"""

with group_driver.session() as session:
    result = session.run(pipeline_query)
    for record in result:
        print(record)




<Record name='pipe' nodePropertySteps=[] featureProperties=[] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [57]:
# feature_columns = [ 'zip', 'latitude', 'longitude', 'blockgroup', 'franchise_bool','medhinc_cy', 'avghinc_cy', 'gini_fy',
#        'indmanu_cy', 'totpop_cy', 'fem25', 'fem30', 'fem35', 'male25',
#        'male30', 'male35', 'crmcytotc', 'di100_cy', 'di150_cy']


add_property_query = """
CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe', 'scaleProperties', {
  nodeProperties: 'avg_rating',
  scaler: 'MinMax',
  mutateProperty:'scaled_rating'
}) YIELD name, nodePropertySteps"""

with group_driver.session() as session:
    result = session.run(add_property_query)
    for record in result:
        print(record)


<Record name='pipe' nodePropertySteps=[{'name': 'gds.scaleProperties.mutate', 'config': {'contextNodeLabels': [], 'mutateProperty': 'scaled_rating', 'scaler': 'MinMax', 'contextRelationshipTypes': [], 'nodeProperties': 'avg_rating'}}]>


In [58]:
select_feature_query = """
CALL gds.alpha.pipeline.nodeRegression.selectFeatures('pipe', ['scaled_rating', 'avg_rating'])
YIELD name, featureProperties"""

with group_driver.session() as session:
    result = session.run(select_feature_query)
    for record in result:
        print(record)


<Record name='pipe' featureProperties=['scaled_rating', 'avg_rating']>


In [59]:
split_query = """
CALL gds.alpha.pipeline.nodeRegression.configureSplit('pipe', {
  testFraction: 0.2,
  validationFolds: 5
}) YIELD splitConfig"""


with group_driver.session() as session:
    result = session.run(split_query)
    for record in result:
        print(record)


<Record splitConfig={'testFraction': 0.2, 'validationFolds': 5}>


In [60]:
add_model_query = """

CALL gds.alpha.pipeline.nodeRegression.addLinearRegression('pipe')
YIELD parameterSpace"""

with group_driver.session() as session:
    result = session.run(add_model_query)
    for record in result:
        print(record)


<Record parameterSpace={'LinearRegression': [{'maxEpochs': 100, 'minEpochs': 1, 'penalty': 0.0, 'patience': 1, 'methodName': 'LinearRegression', 'batchSize': 100, 'tolerance': 0.001, 'learningRate': 0.001}], 'RandomForest': []}>


##### 2.1.2 Train Pipeline

In [61]:
project_graph_query = """
MATCH (bl:BusinessLocation)
WHERE bl.avg_rating IS NOT NULL 
AND bl.avg_rating = bl.avg_rating
RETURN gds.graph.project(
  'myGraph',
  bl,
  null,
  {
    sourceNodeLabels: labels(bl),
    targetNodeLabels: [],
    sourceNodeProperties: bl { .avg_rating, .latitude, .longitude },
    targetNodeProperties: {}
  }
)"""

with group_driver.session() as session:
    result = session.run(project_graph_query)
    for record in result:
        print(record)


<Record gds.graph.project(
  'myGraph',
  bl,
  null,
  {
    sourceNodeLabels: labels(bl),
    targetNodeLabels: [],
    sourceNodeProperties: bl { .avg_rating, .latitude, .longitude },
    targetNodeProperties: {}
  }
)={'graphName': 'myGraph', 'configuration': {'jobId': '52d84b88-3479-4fe8-a636-06b4afb2cb5f', 'query': "\nMATCH (bl:BusinessLocation)\nWHERE bl.avg_rating IS NOT NULL \nAND bl.avg_rating = bl.avg_rating\nRETURN gds.graph.project(\n  'myGraph',\n  bl,\n  null,\n  {\n    sourceNodeLabels: labels(bl),\n    targetNodeLabels: [],\n    sourceNodeProperties: bl { .avg_rating, .latitude, .longitude },\n    targetNodeProperties: {}\n  }\n)", 'inverseIndexedRelationshipTypes': [], 'logProgress': True, 'readConcurrency': 4, 'undirectedRelationshipTypes': []}, 'query': "\nMATCH (bl:BusinessLocation)\nWHERE bl.avg_rating IS NOT NULL \nAND bl.avg_rating = bl.avg_rating\nRETURN gds.graph.project(\n  'myGraph',\n  bl,\n  null,\n  {\n    sourceNodeLabels: labels(bl),\n    targetNodeLabe

In [62]:
train_model_query = """

CALL gds.alpha.pipeline.nodeRegression.train('myGraph', {
  pipeline: 'pipe',
  targetNodeLabels: ['BusinessLocation'],
  modelName: 'nr-pipeline-model',
  targetProperty: 'avg_rating',
  randomSeed: 25,
  concurrency: 1,
  metrics: ['MEAN_SQUARED_ERROR']
}) YIELD modelInfo
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.MEAN_SQUARED_ERROR.train.avg AS avgTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.outerTrain AS outerTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.test AS testScore"""


with group_driver.session() as session:
    result = session.run(train_model_query)
    for record in result:
        print(record)


<Record winningModel={'maxEpochs': 100, 'minEpochs': 1, 'penalty': 0.0, 'patience': 1, 'methodName': 'LinearRegression', 'batchSize': 100, 'tolerance': 0.001, 'learningRate': 0.001} avgTrainScore=13.985154767745334 outerTrainScore=13.985154753818673 testScore=13.992610282478848>


In [63]:

add_context_query = """

MATCH (bl:BusinessLocation)
OPTIONAL MATCH (bl:BusinessLocation)-[r:contained_in]->(bg:BlockGroup) 
RETURN gds.graph.project(
  'bus_in_blockgroup_Graph',
  bl,
  bg,
  {
    sourceNodeLabels: ['bg'],
    targetNodeLabels: ['bl'],
    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},
    targetNodeProperties: bl {.avg_rating, .latitude, .longitude },
    relationshipType: 'contained_in'
  },
  { undirectedRelationshipTypes: ['contained_in'] }
)"""

with group_driver.session() as session:
    result = session.run(add_context_query)
    for record in result:
        print(record)



<Record gds.graph.project(
  'bus_in_blockgroup_Graph',
  bl,
  bg,
  {
    sourceNodeLabels: ['bg'],
    targetNodeLabels: ['bl'],
    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},
    targetNodeProperties: bl {.avg_rating, .latitude, .longitude },
    relationshipType: 'contained_in'
  },
  { undirectedRelationshipTypes: ['contained_in'] }
)={'graphName': 'bus_in_blockgroup_Graph', 'configuration': {'jobId': '33ea4ef1-b9b2-4a9b-ad7e-4e39fcbb7f3b', 'query': "\n\nMATCH (bl:BusinessLocation)\nOPTIONAL MATCH (bl:BusinessLocation)-[r:contained_in]->(bg:BlockGroup) \nRETURN gds.graph.project(\n  'bus_in_blockgroup_Graph',\n  bl,\n  bg,\n  {\n    sourceNodeLabels: ['bg'],\n    targetNodeLabels: ['bl'],\n    sourceNodeProperties: bg {.avghinc_cy, .totpop_cy, .crmcytotc},\n    targetNodeProperties: bl {.avg_rating, .latitude, .longitude },\n    relationshipType: 'contained_in'\n  },\n  { undirectedRelationshipTypes: ['contained_in'] }\n)", 'inverseIndexedRelationshipTypes': 

In [64]:
create_pipe_w_context_query = """
CALL gds.alpha.pipeline.nodeRegression.create('pipe-with-context')"""

with group_driver.session() as session:
    result = session.run(create_pipe_w_context_query)
    for record in result:
        print(record)



<Record name='pipe-with-context' nodePropertySteps=[] featureProperties=[] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [65]:
add_node_query = """
CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe-with-context', 'fastRP', {
  embeddingDimension: 64,
  iterationWeights: [0, 1],
  mutateProperty:'embedding',
  contextNodeLabels: ['bg'],
  randomSeed: 1337
})"""

with group_driver.session() as session:
    result = session.run(add_node_query)
    for record in result:
        print(record)


<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}] featureProperties=[] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [66]:
add_embedding_query = """ 
CALL gds.alpha.pipeline.nodeRegression.selectFeatures('pipe-with-context', ['embedding'])"""

with group_driver.session() as session:
    result = session.run(add_embedding_query)
    for record in result:
        print(record)




<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}] featureProperties=['embedding'] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': []}>


In [67]:
add_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.addRandomForest('pipe-with-context', {numberOfDecisionTrees: 5})"""

with group_driver.session() as session:
    result = session.run(add_model_query)
    for record in result:
        print(record)




<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['bg'], 'mutateProperty': 'embedding'}}] featureProperties=['embedding'] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': [{'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0}]}>


In [68]:
  

train_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.train('bus_in_blockgroup_Graph', {
  pipeline: 'pipe-with-context',
  targetNodeLabels: ['bl'],
  modelName: 'nr-pipeline-model-contextual',
  targetProperty: 'avg_rating',
  randomSeed: 25,
  concurrency: 1,
  metrics: ['MEAN_SQUARED_ERROR']
}) YIELD modelInfo
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.MEAN_SQUARED_ERROR.train.avg AS avgTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.outerTrain AS outerTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.test AS testScore
"""

with group_driver.session() as session:
    result = session.run(train_model_query)
    for record in result:
        print(record)




<Record winningModel={'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0} avgTrainScore=0.5492980842911878 outerTrainScore=0.5304061302681993 testScore=0.3939714285714286>


#### 2.5 Sample Business Location Generation

In [41]:
from shapely.geometry import Point
import geopandas as gpd

In [37]:
SD_crs = 'EPSG:26946'
common_crs = "EPSG:4326"


data_path = '../data/'



In [19]:

business_location_df = pd.read_json("../data/business_location.json")
# business_location_geometry = [Point(xy) for xy in zip(business_location_df.longitude, business_location_df.latitude)]
# business_location_gdf = gpd.GeoDataFrame(business_location_df, crs="EPSG:4326", geometry=business_location_geometry)
# business_location_gdf = business_location_gdf[business_location_gdf['longitude']!=180] #fix these outliers





# boundaries_zip = 'City_and_County_Boundaries.zip'
# boundaries_path = os.path.join(data_path, boundaries_zip)
# boundaries_gdf = gpd.read_file(boundaries_path)
# boundaries_gdf = boundaries_gdf.to_crs(common_crs)

# sd_boundaries_gdf = boundaries_gdf[boundaries_gdf['COUNTY_NAM'] == 'San Diego']
# sd_county_geom = sd_boundaries_gdf.geometry.union_all()




<class 'pandas.core.frame.DataFrame'>
Index: 39593 entries, 0 to 39592
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          39593 non-null  int64  
 1   name        39593 non-null  object 
 2   url         39593 non-null  object 
 3   address     39593 non-null  object 
 4   city        39527 non-null  object 
 5   zip         39593 non-null  int64  
 6   latitude    39593 non-null  float64
 7   longitude   39593 non-null  float64
 8   blockgroup  39556 non-null  float64
 9   categories  39424 non-null  object 
 10  avg_rating  39571 non-null  float64
 11  franchise   39593 non-null  object 
 12  confidence  39593 non-null  float64
 13  reasoning   39593 non-null  object 
 14  geom        39593 non-null  object 
 15  geom_ewkt   39593 non-null  object 
dtypes: float64(5), int64(2), object(9)
memory usage: 5.1+ MB


In [ ]:
bl_sampled_points = sd_boundaries_gdf.sample_points(method='cluster_poisson',size = 10)
m = sd_boundaries_gdf.explore()
bl_sampled_points.explore(m=m, color='red')

In [ ]:
zone_location_df = pd.read_json("../data/zone_location.json")
zone_location_df
